In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('Task_3_and_4_Loan_Data.csv')

def create_fico_buckets_dp(df, num_buckets=5):
    """
    Optimizes FICO score bucket boundaries using Dynamic Programming
    to maximize Log-Likelihood of Probability of Default (PD).
    """
    # 1. Aggregate defaults by unique FICO score
    data = df[['fico_score', 'default']].sort_values('fico_score').reset_index(drop=True)
    fico_counts = data.groupby('fico_score')['default'].agg(['count', 'sum']).reset_index()

    scores = fico_counts['fico_score'].values
    n_counts = fico_counts['count'].values
    k_defaults = fico_counts['sum'].values
    M = len(scores)

    # 2. Pre-calculate Log-Likelihood for any segment [i, j]
    def segment_ll(i, j):
        n = np.sum(n_counts[i:j+1])
        k = np.sum(k_defaults[i:j+1])
        if n == 0 or k == 0 or k == n:
            return 0.0
        p = k / n
        return k * np.log(p) + (n - k) * np.log(1 - p)

    ll_matrix = np.zeros((M, M))
    for i in range(M):
        for j in range(i, M):
            ll_matrix[i, j] = segment_ll(i, j)

    # 3. Dynamic Programming Table Setup
    dp = np.full((num_buckets + 1, M), -np.inf)
    backtrack = np.zeros((num_buckets + 1, M), dtype=int)

    for j in range(M):
        dp[1, j] = ll_matrix[0, j]

    for b in range(2, num_buckets + 1):
        for j in range(b - 1, M):
            for i in range(b - 1, j + 1):
                val = dp[b - 1, i - 1] + ll_matrix[i, j]
                if val > dp[b, j]:
                    dp[b, j] = val
                    backtrack[b, j] = i

    # 4. Reconstruct Optimal Boundaries
    boundaries_idx = []
    curr = M - 1
    for b in range(num_buckets, 0, -1):
        prev = backtrack[b, curr]
        boundaries_idx.append((prev, curr))
        curr = prev - 1

    boundaries_idx.reverse()

    # 5. Format Rating Map Output (Reversed so Rating 1 = Lowest Risk)
    buckets = []
    total_buckets = len(boundaries_idx)
    for idx, (start_i, end_i) in enumerate(boundaries_idx):
        min_fico = scores[start_i]
        max_fico = scores[end_i]
        n_rec = np.sum(n_counts[start_i:end_i+1])
        k_def = np.sum(k_defaults[start_i:end_i+1])
        pd_val = (k_def / n_rec) * 100

        # Rating 1 = Best Credit Score (Lowest PD)
        rating = total_buckets - idx

        buckets.append({
            'Rating': rating,
            'FICO Range': f"{min_fico} - {max_fico}",
            'Total Records': n_rec,
            'Defaults': k_def,
            'PD (%)': f"{pd_val:.2f}%"
        })

    result_df = pd.DataFrame(buckets).sort_values('Rating').reset_index(drop=True)
    return result_df

# Run for 5 Buckets
fico_rating_map = create_fico_buckets_dp(df, num_buckets=5)
print("===== OPTIMAL FICO RATING MAP =====")
print(fico_rating_map.to_string(index=False))

===== OPTIMAL FICO RATING MAP =====
 Rating FICO Range  Total Records  Defaults PD (%)
      1  697 - 850           1657        77  4.65%
      2  641 - 696           3197       336 10.51%
      3  581 - 640           3438       703 20.45%
      4  521 - 580           1407       536 38.10%
      5  408 - 520            301       199 66.11%
